# RepairScope-X — Notebook 20 | Generation-length ablation

**Experiment:** Real MDLM and BD3LM generation at 128 and 256 answer-token budgets, compared with the **previously executed, checksum-verified 64-token baselines** (both baseline archives are embedded). Same frozen 24 problems for each model, same seed and prompting protocol as Notebook 18/19 v6. No repair jobs here.

**Execution design:** One family at a time. Each executable batch has at most **4 generations** and checkpoints each successful generation to Google Drive immediately. There are 6 batches per budget per model (24 short cells). Colab reconnects can resume from any batch without repeating finished trials. A single trial cannot be interrupted safely, so the soft wall-time cutoff is tested between trials, not during model generation.

**Model environment:** Use a fresh Google Colab T4 GPU. Public checkpoints; HF_TOKEN optional. Does **not** upgrade NumPy / SciPy / Pandas / PyTorch; loads official dLLM sampler/model implementation at a pinned Git commit. The same inference path already produced the successful v6 result archives supplied by you.

**Important fairness note:** 128 and 256 steps increase generated-token budget. The MDLM code uses total `steps=64` with 32-token blocks, and the BD3LM code uses `steps_per_block=32`. Thus native compute changes with budget and must be reported rather than assumed equal. Increasing output length is an ablation, **not** an equal-compute comparison. The previous 64-token data have been retained rather than rerun, so timing measurements are across sessions and should be treated cautiously.

Output: `RepairScopeX_NB20_MDLM_Length_results.zip`, `RepairScopeX_NB20_BD3LM_Length_results.zip` and a combined ZIP.

## 0. Repair-safe environment installation (first code cell)

In [ ]:
# Must be first code cell; does not import NumPy into the notebook kernel.
import os, sys, subprocess, pathlib
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'  # public model snapshots
if sys.version_info < (3,10):
    raise RuntimeError('Python 3.10 or newer is required.')
check="import numpy, torch; assert numpy.asarray([1,2]).sum()==3; print('NumPy',numpy.__version__,'Torch',torch.__version__)"
r=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-7000:])
    raise RuntimeError('Scientific Python environment already corrupted. Colab: Runtime > Disconnect and delete runtime; reconnect; do NOT reinstall NumPy inside a live runtime.')
print(r.stdout)
# Narrow, pinned Python interfaces only; never reinstall PyTorch, NumPy, SciPy or Pandas.
packages=['transformers==4.57.0','accelerate==1.11.0',
          'huggingface-hub==0.35.3','tokenizers==0.22.1','safetensors==0.6.2']
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','--no-deps',*packages],check=True)
preflight="import torch,transformers,accelerate,huggingface_hub,tokenizers,safetensors; from transformers import AutoTokenizer; from transformers.processing_utils import Unpack; from transformers.utils import TransformersKwargs; from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask; print('model interface',transformers.__version__)"
r=subprocess.run([sys.executable,'-c',preflight],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-9000:])
    raise RuntimeError('Model library import preflight failed. Start from a fresh runtime and report this error rather than running GPU stages.')
print(r.stdout)

## 1. Runtime, pinned implementation and Drive

In [ ]:
import base64, zlib, json, re, math, ast as pyast, gc, time, random, traceback, hashlib, zipfile, csv, shutil, subprocess, sys, importlib.util, os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'Choose T4 GPU and restart from the beginning.'
assert transformers.__version__=='4.57.0', 'Transformers version mismatch; restart with a fresh runtime.'
GPU=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
assert VRAM >= 7, f'Insufficient GPU memory: {VRAM:.2f} GB'
print('GPU',GPU,'VRAM',round(VRAM,2),'GB','torch',torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/RepairScopeX/RSX20_Length_v1')
DRIVE.mkdir(parents=True,exist_ok=True)
# Verify persistent destination is writable; do not fall back to ephemeral /content.
(DRIVE/'drive_write_test.txt').write_text('ok',encoding='utf-8')
assert (DRIVE/'drive_write_test.txt').read_text()=='ok'
SOURCE_COMMIT='ca176752fbceec49c6b4777a2c18ae88e4eb10ed'
ROOT=Path('/content/rsx_v6_official_repo')
if not (ROOT/'.git').exists():
    ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','init','-q',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'remote','add','origin','https://github.com/ZHZisZZ/dllm.git'],check=True)
try:
    sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
except subprocess.CalledProcessError:
    sha=''
if sha!=SOURCE_COMMIT:
    for retry in range(3):
        try:
            subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','--depth','1','origin',SOURCE_COMMIT],check=True,timeout=240)
            subprocess.run(['git','-C',str(ROOT),'checkout','-q','--force','FETCH_HEAD'],check=True,timeout=80)
            break
        except Exception:
            if retry==2: raise
            time.sleep(2**retry)
sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert sha==SOURCE_COMMIT
print('Official code SHA',sha)

## 2. Official isolated samplers

In [ ]:
# Exact official sampler files, copied BYTE-FOR-BYTE from the pinned upstream commit.
SHIM=Path('/content/rsx_v6_inference_only')
files=[
    'dllm/core/samplers/__init__.py','dllm/core/samplers/base.py',
    'dllm/core/samplers/mdlm.py','dllm/core/samplers/bd3lm.py',
    'dllm/core/samplers/utils.py','dllm/core/schedulers/__init__.py',
    'dllm/core/schedulers/alpha.py','dllm/core/schedulers/kappa.py',
]
for rel in files:
    source=ROOT/rel; destination=SHIM/rel
    assert source.exists(),source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert hashlib.sha256(source.read_bytes()).digest()==hashlib.sha256(destination.read_bytes()).digest()
for rel in ['dllm/__init__.py','dllm/core/__init__.py']:
    f=SHIM/rel;f.parent.mkdir(parents=True,exist_ok=True)
    f.write_text('# Inference namespace, without unrelated training imports.\n')
# Model class from exact official source, no checkpoint `trust_remote_code` needed.
MODEL_FILE=ROOT/'dllm/pipelines/a2d/models/qwen2/modeling_qwen2.py'
assert MODEL_FILE.is_file()
sys.path.insert(0,str(SHIM))
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig, BD3LMSampler, BD3LMSamplerConfig
model_module_name='rsx_v6_a2d_qwen2_source'
if model_module_name not in sys.modules:
    spec=importlib.util.spec_from_file_location(model_module_name,str(MODEL_FILE))
    module=importlib.util.module_from_spec(spec)
    sys.modules[model_module_name]=module
    spec.loader.exec_module(module)
else:
    module=sys.modules[model_module_name]
A2DQwen2Config=module.A2DQwen2Config
A2DQwen2LMHeadModel=module.A2DQwen2LMHeadModel
assert A2DQwen2Config.model_type=='a2d-qwen2'
print('Official samplers:',MDLMSampler.__name__,BD3LMSampler.__name__)

## 3. Embedded original 24-problem cohort

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

## 4. Exact Countdown + diagnostic numeric verifiers

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

## 5. Recover the two 64-token baselines (no GPU work)

In [ ]:
BASELINE_SHA256 = '78b1a078361b2af37f8a3b74c592d60d39d9e19a59afe4c1311c4164fe07dabe'
BASELINE_B64 = 'eNrtfQtz48ax7l9BKb4Ve00pmPdAuXtvJXZy4lNJysfeW6m6WZcMkqCEEkgwBLhaxcf//XT3DF4kSFHQi15ry96VgJmeB6a/6e7p6f7p5G9f//VvJ+c/nczjRTpLihJ/LtLLRVyuV8nJ+YmazeLIWDtmYWyk4kkSSm2ViaSahiaOx7NEyHhqoyhOpuMomkUxlJ8YoWcsYvHsZHQyyRez9BIJL1d5mU/yDOiuio/Msuh0ks/H6SKZnn7QpzzkmoUsgjqzeJ5mt1CO+jc6mefTJLtIp/BkmmXz06v1+Hf/dZMs+Jk6/QrerU7DM/XH028WRblaT8rTaTqbrYs0X5zOp1D8Q3jGairFVQxkYm5lYiMVay4tV3ocz/hMhyrkiZLxONZszOWExVCtyNerSXIBXZ2nMEEnk5gZbRSfjSdJMpHRRI+lMSbmE2bjxNpEJjBdybSpuoxvszye+qYnk1COLQ8Vm2k+s1qG44hPJINpZVOVzKZiBhMqhBzP5DhhXI9ncaJnKrSzmRLcAt3FhaNcnJxzOTrJ1+VFmV8nC/hdw+/jLJ9cn5wLDj0ok6V/WibzZbLyXxZmRI1OLhOglEBXzwV8DTY6WSXLOF3RM6j1T3zKR/i3+AFoJVkyKS/m8ceTczs6iSclzDAWOynjNOMKOoY/qBDnOp1Os4SeXWb5OM5OfvgZ1sIqictkerEuJzAR+MFPWXjK7DsWnUt+HuozWFhCiS/D8DxEMpfLNRR8lxRZHLyT2EC+mlxdfEhW+HWRxhljZ+GXkzUTWL5cxYtilq/mUKBVSp4pcxaeQA8uoX1ci70r7EWWF/FanF3At8Cv46YUv2r1YgZzCt8Mn4rqI8AcNo9t/d3KVRpn/nNXn3I9gXVSYLmQaKYfkot0USarD8midNOTAodmGfQlzrKLJdCEl9STi3g9TUuoCuNORlDuoojnyyy5yFfxJKNlBETHABwXs/QjdMq35l8kH5cZlIRvdutrXFwl8XSV53NfApgKyOEncS34+rN1dlH1H9EIvswMBoYFSuhYUaYTmJhJ/AGWE/T/T007wTLN8vL3AWDNOEvmp5NsDSywCqjf6eLy98EiD5Z5niXT4O9//lOAHYhXaQGzHoyT8iZJFkEMSywtYZZxen8fXCXrFTUZZHFRni7W8zEQhMWVzlL4YZX8a51CwYDm6gwX2SqZ5CtioJ9OCLQm+XpRTvObxTn0Ox27z3zObKjOZfjfxHsw+77PF3dXgcLTuIyLpGyXvJhBN64cofkSX/2/IgnKqyRwfS4CHo0C+C9eTAMbJB+Bg7PbIF9MoFSOUzZdT1yFMl5dJmXA+FnwXQLzAJMSfIB+TGFy0vJqnuB0wOeFBpEPiGIC/9/Ay+DHP3/z9z/89eIPf//+H3/67jz43671//PjGQEEEj45B8oh4Y9f810YAkBbrnEA76Av/S3efyhBWpy/X7xf8Cj4MsD/Lf7W7Svj0Ef3aSex446foM/FdbOy64W4SgEM68du/cJQofJk87EvDJ2fphNH1k1AVX77Dc32RTPgk/PFOst+HvX3htAfFg6uOWPVSEhj5cjISIw4D0fMjABRGKNf3E+Cu1fwj7aRHolQQTV4gWA/4iYacaYjV1sjAWGgnpTGE5PWVtTqH7EeG0XWSjvSBrbIkYiEglZVQwf2kcUs8RAV31SwB9jR3sFuEIkKYKMFDkmfKeiUVpZH+MfACk/i64sPq3h+cTkGXDwLTRgqrjl00hoTIU4+rhSDuLOGrpzk18Dhd7O1vTdX26FMzeQoYKFjBS4O4oUn4+qDmXqWAqpDA8UNQGiKgwCGZCH8BUN4G8CmHuzm/d7yQxnXP97i3G7xNoMiI+5gXfZw1mUqCkeSc00cV/FNVHMZ/qaa34Af4SfblIT/xEgZ3caA4ZSGsKs8k5IJHYE0j3+E7ePXCCTZUIdMhUxq9eL8Ks39t2EzeBseBcLxqzqEXUX4VPwqwoMY9scff1zellc5bJe/Cb5OgHc743m/qAb2NvinG5v64f3C9/4t9P891fwqzibrDBYPVS7W8/cL+AsKwN+fexJfuKJ+uOUGUrxf0G8XHjegZrn6HGpDreUKhNrP26/hIXT8sWFh54a+GxXcLD8AFYSKYO/lGvZjFgGPAqNqQH3cpBUL3UNhhHHsC4oiMbPb7YGnq30/HEWcjZRWpmFzixCAuzYfoUZq3M6vuSPKuOSuLD6RhodmpEEjhI1dIlUo28AVVWCWYw0lJNXTUIFFBhBJSGhNKPgrDLEJKoIPaXTDYIYrYbTUUqFUAO1uw4zmOgKlS4vImPDlUYbfXyzgg+UCkI7hezqxIDwEZxR/MqBB0kNFgw36QOrHswBh4i/JKsEC5S5BgcT9FnjViITSPzMoPAA0rdyIt1o5Fvjwc/eYUgUDENgtp3thQCGTWgNsD9KllfCUxAhg3pZkQRrBJkB1IKalIIBEAVzaFjgUlQcFZAR1VbS3N1jyIWBhpRJWIQpswYSVRoPewBlXXMoXhwl2f2GEDRZG4KM06sNBqrSQTyaPyOEKhK4VAosyh9yvQPRDw4U37rztkHsSIBiiXnD7ROoFiAU7lQLU6lvSQrRDvdgNBEqFYVuvuKOxB7K5CDkIRxpUi5AEjaNndnVvXlcP0DsAje/B6erJOF3JB9r/9rWo5I87WRx4myNXS8/a/bu/PIznO7v5k+z9sp/jT1pjONlifdfQHpsgGv+sAM6T9gBZALblu3i8Zm/T2d6jHt6+c7OPHrzTa2BXzoX4xVgfBLs3CAg2FAUEbPbcb/jsIMVAPpleIA9XC3CJk7SfB+tHGVGlPuwdEoEI7CkhiQNHYWN8BG1ACOs4Gg3+EQgEtSmhMQpU9gPjf6kODNyefeeZQUTVQFRAewGJGhr/j8yW9uCbMx1RABsZyv0KOxdx5H48Rz12AYBH9zcKRINFAJQB1H2sAqdcPBX3I+lHYP+jGNNLWwn8XD4pLpi2XdHQZt3CBXMXLkjL/CkD2QORUA8mHCabCK1HMrR8xCNnphiqLSgWGa3cuaI+ErC4LOb2GhZBnC7OldA78GGzVAcS6GUbCP4Wl2VwFeN2B2NcrMtgmcWLktYrsEZcInOBphbMEmCQ8W3981kQsKD41zqG3XKW52WQzzyFIpjEi2AeXydQOLiESStaL6HeN7MgLYHnrpMCGXOzBDKnq735KhivS/TaoA09uL7cflMkWVZAf1bBZ7DjX+U3wXw9uQqmOTQFfE5kZzD24AqG1Yz0/7aNDfBJDxc/GkuDCof/OQYBQiVfgvD9QKToWBCQ+9Uv679hcGEsl1wwLXcZFwQPteYhwB2ef4rnhwutjbobLrDUfrj4z3gOnDRe3xaBDsrT4ipdlY7fVPi/gB1nwN7AFrdBEgPbTfKiDD7D44PgLzUrwp4KfLOMb9tMp+8h8OdlnDnS2G6rGwBWvhfIj9Bu8AZevwUBPXQnA9/nI7fdNySWMfQHYI0GNgpu8nUGvycIH+Fx+A6EjyDYw16Lmzbu4l4Q57hd4t6rdQhbuW0xrKSzOOJiWNB+AxdK1PVavgDKi/N0GCAlPnLnf02Lhhk2UlDd2JDhaxbCNi7wBCHsEBnGewz4joUMiCPvbcn17CyynEch18BPUgj9/LxnmDB38x6WuoP38qtFMKZdk3FYvTAFsPHN40m8yvMFLF7iuZskvbxKF5ewe4L4D6sCNVpc9QtY65Nr4JJFkMC2nXk5GffQkrZGTydIF4EM0JcVFOEF7Kwr0CCg3UtoYZXE01tiu2mSpcAbt2fBP5A0EBiv0mQG8nWWxB8Sd/p2nZYTfIvSdjq5pmfLq3yRjGj/zdKyzLBiDi9Wwfdl8iGBQZSuMrqhgmwBoyzhR5ICoGbV3aZXAC7/8HJKw9s0C+UIx+JmYVRVXCVzmG6coK0ZbAOSVPcEJNfiAa1gR90HgGkpd9RvD6+LWvdqB9bJm0AeBY7JR0IxYM8QlAj6bRRpgAQhpDIjC2wMWMVAM+DCOtuCZtLpLDUidWozACH0PQTuRgzjqJ7sLHxHUy2HSI+UdCQyBNMUYBr0yaC9Av5o1iNPWC0tN6AHgVxhI/UC+ofh+gD9A0rtBzVkIOKwakXHc7Ra4G9zYPfb4Kt4FY9ge14kpVPl/zNZrW5BZflAVjp5rs71GaoUDYtskihRIKkqfGbUCBSUtstRlhcFtePoY1Og2GeZUxfSmSPg76TgL+mqopyjLBTkq3hxia7eUB01ECwzR3HEkjjiOHp9i2y6BL5I2saO6J44Qw2f4zDeL6jP7Qeu750nfr7az94vGnI4dnp8DBhh1COBhCBRRZGEgY5HofOFQicoEHr63zJLrA7M1P9eouRjQtvzGh2lGoAyaBdpvR4m12hrRWSFIROE7vFt5lKjk1MoudZCGfkSSoU8SKmQdygV3wPfrHKyQhTpx6BMUceAX+bx4jaYxmvY79Ceh69hXyNmh/U7Wa9QPsluqWJ5tUoSej9yaj/WnVyl2RQK1fo/gkB7l+eHeSY/cwdJVXn2OXlx7ndvhnO/At5jhsuRCTlwoQRlJxSg4gjgbTqMjKy29CtX5HGIBw2wZysN/4dcUD0uuK7eM3R01L6yBf5nSlkqjm6WLALiz9ymGAgmITdKRMbdk5B9SpKKNCx6a4zmUjH+EgKFMYcIFOYOLenPoJyQfzDd5AMBeb2AVTQKvk1KZ6fzGzHTwRLEhCkpFmPYHU9BVwDmmFwnzrx/FWczZwH18gS8v4rn4zVAxwq1jr8kvy2CyxyJeTmAwzYLKktD9yovgc8ukRXjvtqwiPMuiQJ0oBXsy0U6JSUq9qIIif1Vn9pSUtVEZWNBHm86MMvzqSPgRJCOofO+oscGWXhOlLHblacV+h+g+wH+bCtXKYZP3c2MyhzzaVlWmBbAxig+CPRsJm61Gm0tapc7VcshovGGaBUAaNjhDaU7lTdvbvTbZwaaNTkDBYopMq0ADvXIIFEIIGAlQKHCc5jnF0GgdweIIFDqLttKkSyv/EHIKpmt0suEbo6CzhHcxAgdVwn9gygAq/0qX8NyIKBAgwpevoX1FeQfEn9sArNYrHHTXuZ4QABPGpMKTn3hHBF3aSxQ3vdpGd86C6sziiSry1s86iSrpTMBNL11RhDc9Z3A0IIKZ3NtKLlutQm1h1lrVN2RQfFqQDf5qrwKPsPjoBRPS6Hj5dUoKG9AvwludvXf1d1sjehuamWdmbmz90USXMZTwLWic/FMhYcZlnHXgBFNqwvCBTqSpsWVO+Dtnu8Kuh8KFb6qzteDyxSUy/0nxusCUZ5MdMV6ucxS6HplgVuAShggxeQj3aMeodJ5m6+d0uo+qD/VdqagFt1JZbHmb8QxoOoDj5tB+GF4/wNEK9CzRnhPno58d7qGoeka76lgPasFwC4HUabfrURrPGam82oOQMmE4e5QGsUyoswYyWASkRid0vA6DUpn7j6Nsw15FxUoiUZsj+bw1g4z9vAQemTweAgdU/pQFuRKYyIdKquVxE3n+YUzKfkBwhmUukM4W8WLSVr8tpLOEG95sL5eZ4AzsLZlcLlOgXkLj6zBJFmVQLplq/6Q5lmKGs+fkJV8VSIk8doXMFn1zpGiV3rjlSOyWatjRGpa9O8RilBOgxJNhzvd9V0js1RYVWupXtV70ry2p2JDNT1YONuaG7d99M11p2e0AcHuASLrqOU48/AhNybr5nRcHofMJx/jzgzBBopkWoAaKAQ6zqPJh3EzQvgB1EG1rpb8yOctCjVZmZnixlRQUpOitxJBiR5LbRCHoJSWHAurSEb1/fym3f4m2obsrkt/9BBx0EYoCUaaHB1ED04JZUIVSmatEkYBfD4LThUfYM/0CDQBBek00qIfq/pKdvCKCrTx6g8zlE5ARsG9O0ZpZVXeBvMkzpxN+AalOdDcisTJLH9cT66hKCh6IK5kMW7rxGOj4DuAnSTDw7HVAnZ+4lCpgH+KFOQ2dJmZJ9WB0BaVIHd3XmG9AsOtAUHiW4Iry6NgmeQgMjhKRLmPAIoxqScSF2VgFdIoGpWx7sh+OpVT0LR7fA9b7H3h6uB2CD2UO8B/B1OdZyQTof+O+7gj+AzkclQ5AM7yLMtv8JNhyB+QKMlbF2b7LU3Y72D0zhtgE6PUcWCUeiSMYiyK3M08QA+UaoQQEQgtEeEVw2OvFjLwsD7IZ1yitKPRis3wXl9EB/fKgkAmde0ByA0e6Esbau+xS0TqC37ChwTRYRMHxNnKyVGgD52oDwPt5dKCwMg5+eDwHpc9DkIfSm1GMSERg18GndiuW3x9JfejE4kyy9z578H2O8tQK6ID/LT0yBOgf54MOy9JwDlrm4vQeWZFzIO+76DOZVmbvVk4QBghqmnRaHWkqk3TD+k09ULFpqRV9dErpNsvoGuJH7QDhPY9HiL2Fgf7frEERRB+ds9+B8+qSADYqy8+OUkEtmTB3F0crawldxsZkW25Osyuy7KIW+eQ05FA3GMRSqchAVTYGg+27vbg9l+bmaLKzIQKk+Sht0BhoxXr+zJ4ykZWbjxl45LxkR3o9CMt0yFjTmeyR+L0sy1fWHaoJGLZfl5nYQCLB71dQY7wJ805iAxkcL5cYXyy+p3jYTp0hlWErrpOs/FoEHu2AL5iqq7VqdFgwx2k2yihBoDEBvkPcZrFME0bvUHr8sHjv/9ona7S7Qo2eRziAFOPCBSaocdLqHDfpiAeeAkXmd6pHXgxpu3NolCrse0KqH+wUKqRDcUWtep0Cx7xkUIDjfZAQEfr3kIdduh12vZKyybdtg19uIQg0FPGWEFe/UIfS7SwLSwAqepA1ICSd0gI6KJHQvR3tyAgF8sED3A0Wq9hNwUhm/QRMkwuLjO0d9JlmZ73E1AfkiJpSQ3Eea5gdZRM9HvJuvOvxJXsodtxvRvgfC9Q3EmJbcfJJEZNoOrPA8fbspE0FMUeincN9ewTsNJuufq7yzssjKpAIsadcQsQH3h9tsXwhAzv+YKoDkICesK3rgl1XyuN1tW2waRLj9leetAFV5LOzbeoDbw7KKLQaOUCCpm+m8NCAG4YHmnQg4ySL4QcIjxUt4CS+5GjUR5g5205A+f5tdt9devpPP+QNpvqb4PJKv73bVCkWQZ/T67yPIOHBSxCvzvj0Qa6EuOhcOV+7Egg5ZsYvYbhZVT7zlGrXeRxz4jZfF06LKlIDxNMPKCQHaYxMJAwEAWn2N23KBYMskVM8mmyGXugDljmyVdaylNGKhskg8jHwgs0Q1Qmhfa2LukaYHOk3Q4MMswwwfGOf29ogogz02lgs/FaURERHeo/LBwZAFEEUgdaY6JI9njzMoAuNE4YriTh1MugBwzwQPSAkvvR4+v1qtLx/7C6htUcO9Z+l3xEv7QET2fI4AkFvov/na/GMV4PcBd5guIqXwKnT5Pgs4CD0D6+Ja9ad6mhuu1z1vH6p7s8gBZkHfBFOjLF4TeK+0iR4/Devjiu3waEjs2jJgEQ8pn6HRok4cFRhBBij+J6ixeMUNFHCwJeLwLVW9v6JlF1HZhYGg0QoMALXd08aq4j4abPQ0UsvcHsjWGjl2B1iUm41/CTbJskq7JDIwiA7GGFJpdcK49WCohkdKjVQUb7+fiv62v0q8BUDLA7ocdcildxKjPcCh3CkFdiYufKC3WZxbdYQRlXonBFYD1+SF3Jtu2RKPpbgcUkbywMWLBzIsojzu/tskbkK9LTNm2n45vgDQ0NN3azcU/wW9qmaecmzxNkb4drLQavrgyeBz8ihePYr3Eoj+RKr0IvUIMAbBt93YbRhl7OeXP3xe3hbGMbhxI7rgsyEXLjtmy6ocgst60tvroxyJW7+98mpwdesuGKCQHbOl2y6QsIJkMNHeI6ipSV0UvZEKU+kJml3s/L/x84NkZrALCaUMFyXVydrpduXxYgsq8XlWXs8nYeTLK4wJv4dNL4dQwbGFVU1ru4VrXH6zKwBpRcKF3TIKn83669TYtBXZMK1VWQeN3DvOOmOsQT4vCmyHqg0F3VIAYAwnQgoBVJEF4dBW9z/ogGQTTjSUmhfI1mTn2WzFCcXgy0ATwsLEc2A3VaK9HW+OHvdox/3vE8dUFBelh9K4a5LznwqhxeuuehpE2Z97AxU2EUwVLRWmn0oXhSLsZkNn/8WuxPngQdHicg8vMZdCoeg1RhIzOzCqQLptDdNjEijpOxmHE1E+hUxawU8HUEZ1MeTR6aPMl18CHZk8ZT0Z/fRo4nMxHD+GKdTLidRpNZNBHJeCJjFavpJGQTHXIxeU2fNCx9Ev5g7pM8SZ0LcxYaKaR6xuRJnRX2MsvrPumT5NOkTwJknqUfTxGqUnxEeZBm8Og1o9JrRqWjy6i005q5SNBLKw8cztTda/WJPLXoasBDp+Bd17e9uKqc24GJJmmRHDj8Kgo83rAKEKtPx7en+G8QL6FlMuzkAfyTJn7E50cQqu2hCZ72GEcZSBBke4XtUDW+8eTJjr5gsEeyraBuD04N5SNCO395rp0Kp8kYe89oblJoFBrhJwo4jxRwECJEY60UqGtGPLQj0FCdjZmhnxdNA1qAa9FSHy5a8rPIasUiySlebCjFtpcJMyG0a5UUYYTdfGwx7zW71ONll2rCunqrDsDdmqLc1XeK3JxtntFMk1lQLZgWX37+BRQM4E8ryU01KVz84N61hvU2mL0/+ckX/mf4w89B/Qtr/8J/+Pn9iavtY1BDve4s/NSQpbLVqdHlUViSHwZiTUIJLb1VSHM8K9KRzzGBqNZ3qmNUiEloMCx9iEbnCM+ENfeQViXAcec7rH1BkrXyXGEmHKzkUuShOovoKRmPfCYd5aLSisjCE7yRXT/VvU+Nf6oxQY/GO+Kho7kD8dBgbUFjRTXZCH8KxbUdCGHaSFD13cVvo8zxQ9hrwq2nwjAS7y58gT70alJ04YtOmq4tLAPY8SCFLadoNaQYOJ9nyaLO2VW1sVH7S5efqwK89Icvgi+BIP73ZVA//ZL9cBTWtpeEMyfM6ZCh234/nu1O6IVlnUDWSerFPBDVCGfRSQafYJ905Q4YWsCmKGQ+u1cUUTyKSpjD+5F4092/RAJSu1vpdC1JOMsgdRXfMgJb36uBUAYSmFDCJfAIfwFI9prU66lVzmI9L4L1EktCm/6G9WGz4sL830M/REAFCfbNm+/JUkTN/N018+bNOZ7/V50umgJNP44ByqIXUy+FFohooVMXu9m78E52R/Ns1E6XF2xPLPEqj8B99UGfDVWEBrHKVFCM+QgjRQ7P6H5Qj0pwr992+zgMx6yW0AMlCce0On4ce8069sQwFk+nFYwJ2YNie+bkeWGM6aOQyPSL4RhjQpsOjvnUZ1sg5iLxbOqZLRzrGNFeDMeom0PlMQWwEhntciTI4wey14xqD0KxWUoROe7EMNWHYbsnZDiEfTPFo7aZuyv6revxBojd2edPQMU8CM+kMQfhmb+ruY1n7QQvzoL2lGDGraCr5cInnuaYVaEP0vYNbKB8pgCVjLGszyvs+EDtNUXcE0tnt2mSTdsnm3jT/gaj2idQYU332rfxbs9MDQe8r5oI1DuEtlaM6qorvxaE26F5KolX0KuL7q1YOQgrCrGmB+zulejuMfDOMsbkHdIbU3i/rd3RgQAHQ4d+Ozsa/yVooK+Z8B4R5eYxBqePybHnACeOYbOx04/Dzw5Qm2R5UUc8cnWH+G10pcFfG9gJFRl+hxvH4+Xm23bmcBEWOV7j1arJtIE1BztrbMmBA2FOSkCvMKJoZTrSRwJzrzn8jiyH353IOQbkvA4ozGvrhDVIF2gkm8Oqwy6jP7APSpJREiMM84rABQoo3v5bQ7nbs+CvSQnYBgtjVbpNgO4ipRhaN12gi63bwmpQ+zqdwwJH59XqKvS3CY0Y5b537qrVYvfnfb/gSOW/1lAoLW+Rxrf++ywx4ZL78n+GokDvE3AeOUwVpmvBkWqdvzIoHGGAawOc7EKcEBEXCQlxCxaUSxKkjaFojBhcjZH7GiGexrsTERDdQjCGnh1VEiGFJ7mAfdx4EQ8hFSQxu50z0QDKdO40Nk81Hj+Yugm8reHDQCF5w9GjhKLcKo1J2mHoXnkefGIRKhNZFwa81w3OYmRawY2IuDL2+ZH0U0lveLcY1xMiu7qwTE+qPo9zTBFSRVKbrIsyn2NY740Miojy6F7bidE2X2dlusxuu0otNVGTv/VgiNG+EUv8m7Pgm4Xr/SQutsLK1tVTnF+Ey14iPnvjUdy24uEzQBILpdnImlghRutWdGh8ZDcKkQS6WyWTNbXwxjQFOqmjwFmJd006kd5aeR89MQHdGnXvbANpo9wVTFBN+VbA2oZIjVxeruyn180m2brtdb9DB4zAIrWKCIqgK8cn1L1me/yEsz0OwufDMzbGFG4YP1wFve2BO+FxQxwlybNlIryz/Ybi+Zs35J932o4U2m7Rv6Uwo/DAESqapXYUWaufHZ4PzQrJOMK4QnNeT7LJRo7syKMkgG7KkNIA0B6QwxIlVhRWudIuoA2Kf6rKk9AtizJkVc6C+Abdx0RVkZS8kjilxev4vuDQw5RIABiHdDvX9Ong0LSxHPYtZqKXUMFf01g+axrLewIoDYksGM24aN6edlyDoXbzk9UBTTtY21/412K7rNGUEmWi2RIAiflIgJRgE3NnMYxGjG7GNjT+6MOijzJiE/myYG4tKylgCKZEUApbDlWTy1y4XOY1UDIrhJNMHwa/gppndZrQMNwC3loo75QdKPKi0zgXoYtr3u9n8/Lq968kEejd+DXFNIPz7hkssnjdg8+//+ILFFTdWAFl3K35s+A/0jp32JVLaCNoGC4ZYXLIrLhSdQzCLpb29GWvru9uat8Gun0sgzOKIRaB/DHg1TO4Alq+GSmd0nIKPNkIR3bkQiIJCleEiUDpuANTwGAUM425POlkBpjeNJFLXbJPBK7qxd7coVWhVqjDBka7/dqp82ODtTmRdY9nKOUo5buSQwU9KyzdNaPLGUoe42nLa4LRI00wOkirbjpIXXCzX+cjHceYGtCnzXEHLHUcjl2qdHWyQ+c2GOcR/j1Us+5+1+orNFLfV+79ebMq/ItW4V+Tf3VXpW7EJLxUhi9lK4Eq1KJAzvVBCyjAfI8Q52LLojyGeOePnQ/TqBlmCvW4bKGKwPBWG8KdEIp+a/t5U/5XullXSYDt6g/1tBZSw8bA2R5sfXEJ8DUP62se1kfIw/rQvaDbHz/y2lmg6tnh+H92oLrf126D/u0v0P/Nu0v9BnaLq7411Kn4qzMYdPcKLtD5UlrhQgkrRnmQ5AjTkSkhjLrfFkEm27uV/r5WN3cI/xQG5/2hOnXI9IGGVhypjXx2E0wyS+L/VoWhXp4WOgEqN1lejTpC0+trTtkjzSk7CIOHj4024MVW5oQtt6h95pWKFMJ128kK4LXq7K6kc767JOmf7lsB8PooEsc9O+Q+NEstuimE5Mq+namhdpzqt7W0Ut1iB8NQt7yvCLuN9p7zPV11Yj93B1yt7mJkndoaE7my5FTFh3pIaQn4ZwlsMVjuUYDta2LcX1Bi3HtibivNphvlzVU+bKR17oqpT3qz1yi8o7XtJqp53Erl2aJQLZLDP21jiT4GJLbP6nzQZOrkCqCSjrT68vtya5xTKoiUpp1417qjrzq1Bkbzll4E7jEZ9zSz0cAG7e2UohUFEQlT3w7YzEVMXdUoNlNC4pZdeqC5hClhIuVinkVKH4u55FeVCXjwORn14PMxNFt84Q04zkWVAlBPN73mt1IEO2vOdhZi5wTnwWY7sTA56Tfq+HYBdE/FrMONgDlNFvkWGm8Ux0O5H/2zC3h2gc/eBvJXIkduy3M+UTF6F5H/u6XTsdptVTLON5zqu/mJvcdrT+JjEANljTGbtYwx0a6Ux3W2466EaWyo5d7yZFHA2LUo2wFgaQz5CMjTyoMQDZUnOc4nxoGMLArHx3Ga9prf+EmhbyMBcae/PedZTcXNievTcnvJ3tcQ+pvf/KYKtBH8EetgFW8f/S7BQzhqBC8nue95FCbK8CWAbiuFcTfH8taZVuukfjvhck8S950ZlO9t85TegR9JKI2tYRBdSoHYmEN1KIS7XuQadbeXummih97TFAbnkcQ1YcSxYt1rVuZD0O/RBvSujZNXPaNrkjEPGVltbSzWY/h0nUBnzunBt1HZJSrCdJZ0R9lqEC2/KbqEVRki/Q3PVnxeInHh2/gELmJKK+WjJH/mIZMOXclHifyiKGUVnfbUCaHvlQS6sUIKjkc2XYNjRThCfyhHToVyXyFHtnvfnW7Bo9bbG9tXhFga4/NGbHAYDwXgFlp3FYqLo8XN15zUD8lJPdgs2EyK78mNc3p1Y/RS4sIFSMOa1cRNkjo7D4mRN3kdrwhR+Ka2FaZl5Rv11d4udNrda1vc6Gg15Vsq+TZdvOj+a7m+vm0P1Bh6G6E0wojghG0MBUEZuavseF7j4qThvVFt0FNJ1kGRbCeAEaFsBY9kIBS87T2F+dX6mu+0usOg2O2eEhjCfEts3iLmLrsPNAVyZQC5FMmWMORjtQS+Zt7en3kbs1M7I+YsX/lb5YBc6IHj0nNWOXipaPAmUJTKUx1Jvs5nSK1dZ8FGA7/ixtQXthX6N2LKzTqJbqe8aCJMdLN3Eq2hFngLuqgO6TjUHq1k8ponG8QLz0AkX/xrnRQl+bIf7Ovhu0Xj78mw3TFbtQxWbrSVrWqLHFrPqcydssfXyWzbkFaRp/TeLghOb7nN1qDGpyJFEMc7IaJxvLjTv4NyfXMpUd3qyfbdNlq1jOJKNJEfuqQQq0DwuJ+IwWzl0bHRBlqzOrnEmzg4PXW6HYC6A9UsYySQE848dTQXW17ThA9NEz7c0aI9E+0xlEFrzjac2XaYl2paPeakhnTeqGf+9Z6pOcT17eDBHAMMujfPrk21kqGTF1udC53Cydap0Ckzeo9jW59JqaHYY0mqyDfRrKv3+1Oz3+kPd48RDdauGCbc42iBevGwYD/8/PP/AO1qIyo='
raw = zlib.decompress(base64.b64decode(BASELINE_B64))
assert hashlib.sha256(raw).hexdigest() == BASELINE_SHA256
BASELINES = json.loads(raw.decode('utf-8'))
for family in ('MDLM','BD3LM'):
    info = BASELINES[family]
    assert info['manifest']['config']['n_sources'] == len(SOURCES) == 24
    assert info['manifest']['config']['source_payload_sha'] == PAYLOAD_SHA
    assert info['manifest']['config']['out_tokens'] == 64
    assert len(info['records']) == 24 and all(r['status']=='ok' for r in info['records'])
    assert {r['problem_id'] for r in info['records']} == {s['problem_id'] for s in SOURCES}
    print(family, '64-token baseline:',sum(int(r['task_success']) for r in info['records']),'/24 success')
print('64-token baseline integrity: PASS')

## 6. Robust model loader, checkpoint writer, and generation engine

In [ ]:
PROTOCOL='rsx20-length-v1-20261019'
MODELS={
    'MDLM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1',
        sha='a284e895a6248256baf2f60502e54aba61b24c1a',
        actions=['tail25','tail50','middle25','global'],
        sampler_cls=MDLMSampler,config_cls=MDLMSamplerConfig,
    ),
    'BD3LM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1',
        sha='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=BD3LMSampler,config_cls=BD3LMSamplerConfig,
    ),
}
OUT_LEN=64; BLOCK=32; STEPS=64; TEMPERATURE=.15
GEN_SEED=38191; REPAIR_SEEDS=[38192,38193]; SELECT_MAX=8
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each supplied number once.'
CTX=None
SRC_MAP={s['problem_id']:s for s in SOURCES}
assert len(SRC_MAP)==24

def normalized_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,torch.Tensor):return x.detach().cpu().tolist()
    if hasattr(x,'item'):
        try:return x.item()
        except Exception:pass
    raise TypeError('Nonserializable '+repr(type(x)))

def atomic_json(path,value):
    path=Path(path);p=path.with_suffix(path.suffix+'.tmp')
    with p.open('w',encoding='utf-8') as f:
        json.dump(value,f,indent=2,ensure_ascii=False,default=normalized_json,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(p,path)

def append_record(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,default=normalized_json,allow_nan=False)+'\n')
        f.flush();os.fsync(f.fileno())

def completed_map(path,signature):
    records={}
    if not Path(path).exists():return records
    with open(path,encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except Exception as ex:raise ValueError(f'Malformed checkpoint JSONL {path}:{lineno}; backup before editing') from ex
            if r['signature']!=signature:raise ValueError('Checkpoint run-signature mismatch: '+str(path))
            if r['id'] in records:raise ValueError('Duplicate trial ID in checkpoint: '+str(r['id']))
            records[r['id']]=r
    return records

def model_dir(family):
    f=DRIVE/family
    f.mkdir(parents=True,exist_ok=True)
    return f

def config_signature(family):
    m=MODELS[family]
    frozen=dict(protocol=PROTOCOL,family=family,model_id=m['repo'],model_sha=m['sha'],
                source_commit=SOURCE_COMMIT,source_payload_sha=PAYLOAD_SHA,
                n_sources=len(SOURCES),out_tokens=OUT_LEN,block=BLOCK,steps=STEPS,
                temperature=TEMPERATURE,gen_seed=GEN_SEED,repair_seeds=REPAIR_SEEDS,
                select_max=SELECT_MAX,actions=m['actions'],budgets=[128,256],baseline_sha=BASELINE_SHA256)
    return hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),frozen

def files_for(family):
    folder=model_dir(family)
    return {'folder':folder,'manifest':folder/'run_manifest.json',
            'natural':folder/'length_trials.jsonl',
            'repair':folder/'repair_trials.jsonl',
            'cohort':folder/'frozen_failure_ids.json',
            'errors':folder/'errors.jsonl'}

def init_family(family):
    paths=files_for(family);sig,cfg=config_signature(family)
    if paths['manifest'].exists():
        old=json.loads(paths['manifest'].read_text())
        if old['signature']!=sig:raise RuntimeError('Different protocol found for '+family+'; do not overwrite existing run. Rename Drive folder or restore previous notebook.')
    else:
        atomic_json(paths['manifest'],dict(signature=sig,config=cfg,
            created_utc=datetime.now(timezone.utc).isoformat(),gpu=GPU,torch_version=torch.__version__,transformers_version=transformers.__version__))
    return paths,sig

def unload_model():
    global CTX
    if CTX is not None:
        print('Unloading',CTX['family'])
        CTX['sampler']=None
        CTX['model']=None
        CTX['tokenizer']=None
        CTX=None
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_family(family):
    global CTX
    assert family in MODELS
    if CTX and CTX['family']==family:
        print(family,'already loaded');return
    unload_model()
    m=MODELS[family]; paths,sig=init_family(family)
    print('Loading',family,'public Hugging Face checkpoint (no token needed)',m['repo'])
    # HF cache-first; otherwise pinned full-snapshot download; simple retry.
    try:
        local=snapshot_download(repo_id=m['repo'],revision=m['sha'],local_files_only=True)
    except Exception:
        for trial in range(3):
            try:
                local=snapshot_download(repo_id=m['repo'],revision=m['sha'],max_workers=4)
                break
            except Exception:
                if trial==2:raise
                time.sleep(2**trial)
    local=Path(local)
    assert (local/'config.json').exists() and (local/'model.safetensors').exists()
    try:
        tokenizer=AutoTokenizer.from_pretrained(str(local),local_files_only=True,trust_remote_code=False,use_fast=True)
        if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
        if tokenizer.bos_token_id is None:tokenizer.bos_token=tokenizer.pad_token
        if tokenizer.mask_token_id is None:
            tokenizer.mask_token='<|mask|>'
        mask_id=int(tokenizer.mask_token_id)
        pad_id=int(tokenizer.pad_token_id)
        eos_id=int(tokenizer.eos_token_id)
        config=A2DQwen2Config.from_pretrained(str(local),local_files_only=True)
        if config.model_type!='a2d-qwen2':raise ValueError('Wrong checkpoint architecture: '+str(config.model_type))
        model=A2DQwen2LMHeadModel.from_pretrained(
            str(local),config=config,dtype=torch.float16,
            local_files_only=True,low_cpu_mem_usage=True,attn_implementation='sdpa',
        ).to('cuda').eval()
        nrows=int(model.get_input_embeddings().weight.shape[0])
        assert 0<=mask_id<nrows and mask_id not in [pad_id,eos_id],('Mask token missing from trained vocabulary',mask_id,nrows)
        sampler=m['sampler_cls'](model=model,tokenizer=tokenizer)
        CTX=dict(family=family,model=model,tokenizer=tokenizer,sampler=sampler,
                 mask=mask_id,eos=eos_id,pad=pad_id,paths=paths,signature=sig,
                 model_sha=m['sha'],model_id=m['repo'],classes=m)
        print('MODEL LOADED',family,'GPU allocated GB',round(torch.cuda.memory_allocated()/2**30,3),'mask id',mask_id)
    except Exception as exc:
        append_record(paths['errors'],dict(stage='model_load',signature=sig,error=repr(exc),traceback=traceback.format_exc(),timestamp=datetime.now(timezone.utc).isoformat()))
        raise

def prepare_ids(s):
    t=CTX['tokenizer']
    msg=[{'role':'system','content':'You solve quantitative reasoning tasks carefully.'},
         {'role':'user','content':s['prompt']+PROMPT_SUFFIX}]
    ids=t.apply_chat_template(msg,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    ids=[int(i) for i in ids]
    assert ids and CTX['mask'] not in ids
    return ids

def visible_answer_tokens(token_ids):
    result=[]
    for i in token_ids:
        if int(i) in [CTX['eos'],CTX['pad']]:break
        result.append(int(i))
    return result

def decode(token_ids):
    return CTX['tokenizer'].decode([int(i) for i in token_ids],skip_special_tokens=True).strip()

def sampler_config(answer_budget):
    a=dict(max_new_tokens=int(answer_budget),steps=STEPS,block_size=BLOCK,
           temperature=TEMPERATURE,remasking='low_confidence',return_dict=True,
           right_shift_logits=False)
    if CTX['family']=='BD3LM':a['steps_per_block']=32
    return CTX['classes']['config_cls'](**a)

class CountCalls:
    def __init__(self,module):self.module=module;self.count=0;self.h=None
    def cb(self,*args):self.count+=1
    def __enter__(self):self.count=0;self.h=self.module.register_forward_pre_hook(self.cb);return self
    def __exit__(self,*args):self.h.remove();self.h=None

def set_seed(seed):
    torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed));random.seed(int(seed))

@torch.no_grad()
def real_generate(prompt,answer_budget,seed):
    set_seed(seed)
    t=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(prompt)],sampler_config(answer_budget),return_dict=True)
    raw=result.sequences[0].detach().cpu().tolist()
    if CTX['family']=='MDLM':
        assert raw[:len(prompt)]==list(prompt), 'MDLM changed observed prompt'
        answer=raw[len(prompt):]
    else:
        offset=math.ceil(len(prompt)/BLOCK)*BLOCK
        assert raw[offset-len(prompt):offset]==list(prompt),'BD3LM changed observed prompt'
        answer=raw[offset:]
    assert not any(t==CTX['mask'] for t in answer), 'Unresolved generation mask(s)'
    assert len(answer)<=answer_budget
    return dict(token_ids=visible_answer_tokens(answer),nfe=int(calls.count),
                raw_generated_tokens=len(answer),wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))


# NB20 budget-specific seeds are the same fixed source generation seed used by v6.
BUDGETS=[128,256]
assert len(SOURCES)==24 and Counter(s['dataset'] for s in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}

def one_length_trial(source,budget):
    assert int(budget) in BUDGETS
    ret=real_generate(prepare_ids(source),budget,GEN_SEED)
    text=decode(ret['token_ids'])
    verification=verify(text,source)
    cap=int(ret['raw_generated_tokens'])
    return dict(id=f"{source['problem_id']}|{budget}|{GEN_SEED}",problem_id=source['problem_id'],
        dataset=source['dataset'],target=float(source['target']),prompt=source['prompt'],
        budget=int(budget),generation_seed=GEN_SEED,output=text,verification=verification,
        task_success=bool(verification['task_success']),strict_success=bool(verification['strict_success']),
        numeric_success=bool(verification['numeric_success']),visible_answer_tokens=len(ret['token_ids']),
        budget_exhaustion_proxy=bool(cap==budget and len(ret['token_ids'])>=budget-2),**ret)

def length_progress():
    assert CTX is not None
    done=completed_map(CTX['paths']['natural'],CTX['signature'])
    for b in BUDGETS:
        print(f"{CTX['family']} / {b} tokens: {sum(x['budget']==b and x['status']=='ok' for x in done.values())}/24")

def length_batch(budget, start, count=4, max_minutes=8):
    assert CTX is not None, 'Load a family first.'
    assert budget in BUDGETS and 0<=start<24 and 1<=count<=4 and start+count<=24
    path=CTX['paths']['natural'];sig=CTX['signature'];finished=completed_map(path,sig)
    started=time.monotonic()
    for source in SOURCES[start:start+count]:
        trial_id=f"{source['problem_id']}|{budget}|{GEN_SEED}"
        if trial_id in finished:
            assert finished[trial_id]['status']=='ok', 'Failed prior trial must be fixed.'
            print('SKIP complete:',trial_id);continue
        if time.monotonic()-started>max_minutes*60:
            print('Soft time cutoff reached. Rerun same cell to resume.');break
        try:
            row=one_length_trial(source,budget)
            row['signature']=sig;row['status']='ok'
            append_record(path,row)
            print('SAVED',trial_id,'correct=',row['task_success'],'NFE=',row['nfe'],'seconds=',round(row['wall_seconds'],1))
        except Exception as ex:
            append_record(CTX['paths']['errors'],dict(stage='length_generation',family=CTX['family'],budget=budget,
                  id=trial_id,error=repr(ex),traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Trial failed; completed checkpoints are safe. See Drive errors.jsonl.') from ex
    length_progress()

## 7. MDLM | load and real CUDA smoke test

In [ ]:
load_family('MDLM')
src=SOURCES[0]
test=real_generate(prepare_ids(src),16,38420)
assert test['nfe']>0 and len(test['token_ids'])<=16
print('MDLM CUDA smoke: PASS | NFE',test['nfe'])
length_progress()

### MDLM: 128-token budget (6 resumable batches × 4 problems)

In [ ]:
length_batch(128,0,count=4,max_minutes=8)

In [ ]:
length_batch(128,4,count=4,max_minutes=8)

In [ ]:
length_batch(128,8,count=4,max_minutes=8)

In [ ]:
length_batch(128,12,count=4,max_minutes=8)

In [ ]:
length_batch(128,16,count=4,max_minutes=8)

In [ ]:
length_batch(128,20,count=4,max_minutes=8)

### MDLM: 256-token budget (6 resumable batches × 4 problems)

In [ ]:
length_batch(256,0,count=4,max_minutes=8)

In [ ]:
length_batch(256,4,count=4,max_minutes=8)

In [ ]:
length_batch(256,8,count=4,max_minutes=8)

In [ ]:
length_batch(256,12,count=4,max_minutes=8)

In [ ]:
length_batch(256,16,count=4,max_minutes=8)

In [ ]:
length_batch(256,20,count=4,max_minutes=8)

## 8. Finish MDLM: statistical summaries and export (no model inference)

In [ ]:
from math import sqrt

def wilson_interval(hits,n,z=1.959963984540054):
    if n==0:return [None,None]
    p=hits/n;den=1+z*z/n;mid=(p+z*z/(2*n))/den
    err=z*math.sqrt((p*(1-p)+z*z/(4*n))/n)/den
    return [max(0.,mid-err),min(1.,mid+err)]

def csv_write(path,rows):
    if not rows:return
    columns=list(rows[0].keys())
    with open(path,'w',newline='',encoding='utf-8') as fh:
        w=csv.DictWriter(fh,fieldnames=columns,extrasaction='ignore');w.writeheader();w.writerows(rows)

def summarize_family(family):
    paths,sig=init_family(family)
    new=completed_map(paths['natural'],sig)
    expected={f"{s['problem_id']}|{budget}|{GEN_SEED}" for s in SOURCES for budget in BUDGETS}
    assert set(new)==expected, f"{family}: missing {len(expected-set(new))} trials; rerun incomplete batches."
    assert all(r['status']=='ok' and r['signature']==sig for r in new.values())
    baseline=BASELINES[family]['records']
    rows=[]
    for rec in baseline:
        row=dict(rec)
        row.update(budget=64,baseline_reused=True,visible_answer_tokens=len(rec['token_ids']),
                   strict_success=bool(rec['verification']['strict_success']),
                   numeric_success=bool(rec['verification']['numeric_success']),
                   budget_exhaustion_proxy=bool(rec['raw_generated_tokens']==64 and len(rec['token_ids'])>=62))
        rows.append(row)
    for r in new.values():rows.append(dict(r,baseline_reused=False))
    assert len(rows)==72
    by_budget=[]
    for budget in (64,128,256):
        group=[r for r in rows if r['budget']==budget]
        hits=sum(bool(r['task_success']) for r in group)
        ci=wilson_interval(hits,24)
        by_budget.append(dict(family=family,budget=budget,n=24,task_successes=hits,task_success_rate=hits/24,
             wilson95_low=ci[0],wilson95_high=ci[1],strict_hits=sum(bool(r['strict_success']) for r in group),
             numeric_hits=sum(bool(r['numeric_success']) for r in group),
             budget_exhaustion_proxy_rate=sum(int(r['budget_exhaustion_proxy']) for r in group)/24,
             mean_visible_answer_tokens=sum(r['visible_answer_tokens'] for r in group)/24,
             mean_nfe=sum(r['nfe'] for r in group)/24,
             mean_wall_seconds=sum(r['wall_seconds'] for r in group)/24))
    paired=[]
    problem_ids=[s['problem_id'] for s in SOURCES]
    for a,b in [(64,128),(128,256),(64,256)]:
        scores={budget:{r['problem_id']:int(r['task_success']) for r in rows if r['budget']==budget} for budget in [a,b]}
        win=sum(scores[b][p]>scores[a][p] for p in problem_ids)
        lose=sum(scores[b][p]<scores[a][p] for p in problem_ids)
        ties=24-win-lose
        discord=win+lose
        pval=1.0 if discord==0 else min(1.0,2*sum(math.comb(discord,k) for k in range(0,min(win,lose)+1))/2**discord)
        paired.append(dict(family=family,old_budget=a,new_budget=b,improved=win,worsened=lose,tied=ties,
            net_success_points=(win-lose)/24,paired_exact_two_sided_p=pval))
    out=paths['folder'];csv_write(out/'length_summary.csv',by_budget);csv_write(out/'paired_length_tests.csv',paired)
    exp_json=out/'all_lengths_72.jsonl'
    exp_json.write_text(''.join(json.dumps(x,ensure_ascii=False,default=normalized_json)+'\n' for x in sorted(rows,key=lambda x:(x['budget'],x['problem_id']))),encoding='utf-8')
    gate={'family':family,'complete':True,'baseline_64_reused':24,'new_real_generations':48,
       'all_budgeted_generations':72,'budgets':[64,128,256],
       'note':'64-token baseline reproduced from user-provided v6, new 128/256 conditions use official sampler; unequal compute.'}
    atomic_json(out/'GATE.json',gate)
    files_to_zip=[paths['manifest'],paths['natural'],out/'length_summary.csv',out/'paired_length_tests.csv',exp_json,out/'GATE.json']
    hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in files_to_zip}
    atomic_json(out/'artifact_manifest.json',hashes)
    files_to_zip.append(out/'artifact_manifest.json')
    target=Path('/content')/f'RepairScopeX_NB20_{family}_Length_results.zip'
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in files_to_zip:z.write(f,f.name)
    print('COMPLETE',family,'\n',by_budget,'\n',paired,'\n ZIP',target)
    return target

MDLM_ZIP=summarize_family('MDLM')
from google.colab import files
files.download(str(MDLM_ZIP))
unload_model()

## 9. BD3LM | load and real CUDA smoke test

In [ ]:
load_family('BD3LM')
src=SOURCES[0]
test=real_generate(prepare_ids(src),16,38421)
assert test['nfe']>0 and len(test['token_ids'])<=16
print('BD3LM CUDA smoke: PASS | NFE',test['nfe'])
length_progress()

### BD3LM: 128-token budget (6 resumable batches × 4 problems)

In [ ]:
length_batch(128,0,count=4,max_minutes=8)

In [ ]:
length_batch(128,4,count=4,max_minutes=8)

In [ ]:
length_batch(128,8,count=4,max_minutes=8)

In [ ]:
length_batch(128,12,count=4,max_minutes=8)

In [ ]:
length_batch(128,16,count=4,max_minutes=8)

In [ ]:
length_batch(128,20,count=4,max_minutes=8)

### BD3LM: 256-token budget (6 resumable batches × 4 problems)

In [ ]:
length_batch(256,0,count=4,max_minutes=8)

In [ ]:
length_batch(256,4,count=4,max_minutes=8)

In [ ]:
length_batch(256,8,count=4,max_minutes=8)

In [ ]:
length_batch(256,12,count=4,max_minutes=8)

In [ ]:
length_batch(256,16,count=4,max_minutes=8)

In [ ]:
length_batch(256,20,count=4,max_minutes=8)

## 10. Finish BD3LM and package cross-model evidence

In [ ]:
BD3LM_ZIP=summarize_family('BD3LM')
files.download(str(BD3LM_ZIP))
unload_model()
combined=Path('/content/RepairScopeX_NB20_Length_Combined_results.zip')
with zipfile.ZipFile(combined,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for family in ('MDLM','BD3LM'):
        p=DRIVE/family
        for f in sorted(p.glob('*')):
            if f.is_file(): z.write(f,f'{family}/{f.name}')
print('Combined evidence archive:',combined, 'MB',round(combined.stat().st_size/2**20,2))
files.download(str(combined))

### Analysis safeguards
- The 64-token comparison arm is **measured data** from the earlier successful v6 run, not a simulated baseline.
- The 128- and 256-token arms require real GPU inference; do not interpret results unless all 48 new trials per model are complete.
- The `budget_exhaustion_proxy` is not a ground-truth truncation label. A human should inspect difficult cases.
- Countdown requires a valid expression using exactly the supplied numbers. Word-problem last-number scoring remains a **diagnostic heuristic** until Notebook 22/manual adjudication.
- P-values are exploratory, paired over the 24 matched problem IDs. They are not evidence of practical significance by themselves.